In [4]:
"""
surfrad_preprocess.py
=====================
Preprocesses raw SURFRAD 1-minute .dat files for validation of
a clear-sky solar radiation dataset (r.sun CONUS product).

Confirmed file format (from bon15015.dat inspection):
  Line 1: station name (e.g. " Bondville")
  Line 2: lat lon elev_m "m version N" (e.g. "40.05 -88.37 213 m version 1")
  Lines 3+: 48 space-separated columns, 1440 lines per day (1-min data)

Column order (48 total):
  year jday month day hour min dt zen
  dw_solar qc  uw_solar qc  direct_n qc  diffuse qc
  dw_ir qc  dw_casetemp qc  dw_dometemp qc
  uw_ir qc  uw_casetemp qc  uw_dometemp qc
  uvb qc  par qc
  netsolar qc  netir qc  totalnet qc
  temp qc  rh qc  windspd qc  winddir qc  pressure qc

Output CSVs (ready for R analysis):
  1. surfrad_daily_clearsky.csv    all valid clear-sky daily totals per station/year/doy
  2. surfrad_multiyear_means.csv   mean per station x doy (compare against r.sun)
  3. surfrad_data_availability.csv coverage matrix

Usage:
  python surfrad_preprocess.py
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# USER CONFIG
# ============================================================
SURFRAD_DIR = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\surfrad_data"
OUTPUT_DIR  = r"C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\surfrad_processed_0.30"

# Station info
STATIONS = {
    "bon": {"name": "Bondville",      "state": "IL", "lat":  40.0519, "lon":  -88.3731, "elev_m": 213,  "dir": "Bondville_IL"},
    "fpk": {"name": "Fort Peck",      "state": "MT", "lat":  48.3078, "lon": -105.1017, "elev_m": 634,  "dir": "Fort_Peck_MT"},
    "gwn": {"name": "Goodwin Creek",  "state": "MS", "lat":  34.2547, "lon":  -89.8729, "elev_m": 98,   "dir": "Goodwin_Creek_MS"},
    "tbl": {"name": "Table Mountain", "state": "CO", "lat":  40.1249, "lon": -105.2370, "elev_m": 1689, "dir": "Table_Mountain_CO"},
    "dra": {"name": "Desert Rock",    "state": "NV", "lat":  36.6237, "lon": -116.0195, "elev_m": 1007, "dir": "Desert_Rock_NV"},
    "psu": {"name": "Penn State",     "state": "PA", "lat":  40.7200, "lon":  -77.9300, "elev_m": 376,  "dir": "Penn_State_PA"},
    "sxf": {"name": "Sioux Falls",    "state": "SD", "lat":  43.7343, "lon":  -96.6233, "elev_m": 473,  "dir": "Sioux_Falls_SD"},
}

# Your 12 representative DOYs
TARGET_DOYS = [15, 45, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349]

# Years to process
YEARS = list(range(2015, 2025))

# Thresholds
ZEN_MAX          = 80.0   # max solar zenith angle for daytime (degrees)
DIFFUSE_FRAC_MAX = 0.30   # clear-sky: diffuse/GHI < 30%
MIN_GHI          = 10.0   # min GHI W/m² to include
MIN_CLEARSKY_MIN = 30     # min clear-sky minutes for valid daily total
MISSING_VAL      = -9999.9

# Column names — 48 total matching confirmed file format
COLS = [
    "year","jday","month","day","hour","min","dt","zen",
    "dw_solar","qc_dwsolar",
    "uw_solar","qc_uwsolar",
    "direct_n","qc_direct_n",
    "diffuse","qc_diffuse",
    "dw_ir","qc_dwir",
    "dw_casetemp","qc_dwcasetemp",
    "dw_dometemp","qc_dwdometemp",
    "uw_ir","qc_uwir",
    "uw_casetemp","qc_uwcasetemp",
    "uw_dometemp","qc_uwdometemp",
    "uvb","qc_uvb",
    "par","qc_par",
    "netsolar","qc_netsolar",
    "netir","qc_netir",
    "totalnet","qc_totalnet",
    "temp","qc_temp",
    "rh","qc_rh",
    "windspd","qc_windspd",
    "winddir","qc_winddir",
    "pressure","qc_pressure",
]

# ============================================================
# FILE READER
# ============================================================

def read_surfrad_file(filepath):
    """
    Read one SURFRAD daily .dat file.
    Confirmed format: 2 header lines, then 48-column space-separated data.
    Returns DataFrame or None.
    """
    try:
        df = pd.read_csv(
            filepath,
            skiprows=2,          # skip station name + lat/lon line
            sep=r'\s+',          # any whitespace separator
            names=COLS,
            na_values=["-9999.9"],
        )
        # Must have expected columns
        if len(df.columns) != len(COLS):
            return None
        return df
    except Exception:
        return None

# ============================================================
# PROCESSING PIPELINE
# ============================================================

def process_file(filepath, station_code, year, doy):
    """
    Full pipeline for one station x year x DOY file.
    Returns dict with daily summary or None if insufficient data.
    """

    # --- Read ---
    df = read_surfrad_file(filepath)
    if df is None or len(df) == 0:
        return None

    # --- QC filter: keep only QC=0 for our three solar variables ---
    df = df[
        (df["qc_dwsolar"]  == 0) &
        (df["qc_direct_n"] == 0) &
        (df["qc_diffuse"]  == 0) &
        (df["dw_solar"].notna()) &
        (df["direct_n"].notna()) &
        (df["diffuse"].notna())
    ].copy()

    if len(df) == 0:
        return None

    # --- Daytime filter: zen < ZEN_MAX ---
    df_day = df[df["zen"] < ZEN_MAX].copy()
    n_daytime = len(df_day)
    if n_daytime == 0:
        return None

    # --- Derived variables ---
    df_day["zen_rad"]     = np.radians(df_day["zen"])
    df_day["cos_zen"]     = np.cos(df_day["zen_rad"])
    df_day["beam_horiz"]  = df_day["direct_n"] * df_day["cos_zen"]

    # Best GHI per SURFRAD README: use DNI*cos(zen) + diffuse
    df_day["best_ghi"]    = df_day["beam_horiz"] + df_day["diffuse"]

    # Diffuse fraction for clear-sky detection
    df_day["diffuse_frac"] = np.where(
        df_day["best_ghi"] > MIN_GHI,
        df_day["diffuse"] / df_day["best_ghi"],
        np.nan
    )

    # --- Clear-sky filter ---
    df_cs = df_day[
        (df_day["diffuse_frac"] < DIFFUSE_FRAC_MAX) &
        (df_day["best_ghi"]    > MIN_GHI) &
        (df_day["beam_horiz"]  >= 0)
    ].copy()

    n_clearsky = len(df_cs)
    if n_clearsky < MIN_CLEARSKY_MIN:
        return None

    clearsky_frac = n_clearsky / n_daytime

    # --- Integrate to daily Wh/m² (sum W/m² × 1min × 1hr/60min) ---
    glob_rad = df_cs["best_ghi"].sum()    / 60.0
    beam_rad = df_cs["beam_horiz"].sum()  / 60.0
    diff_rad = df_cs["diffuse"].sum()     / 60.0
    dw_solar = df_cs["dw_solar"].sum()    / 60.0  # raw GHI for reference

    # Mean zenith during clear-sky period (diagnostic)
    zen_mean = df_cs["zen"].mean()

    return {
        "station"        : station_code,
        "station_name"   : STATIONS[station_code]["name"],
        "state"          : STATIONS[station_code]["state"],
        "lat"            : STATIONS[station_code]["lat"],
        "lon"            : STATIONS[station_code]["lon"],
        "elev_m"         : STATIONS[station_code]["elev_m"],
        "year"           : year,
        "doy"            : doy,
        "n_daytime_min"  : n_daytime,
        "n_clearsky_min" : n_clearsky,
        "clearsky_frac"  : round(clearsky_frac, 4),
        "zen_mean_deg"   : round(zen_mean, 2),
        "glob_rad_Whm2"  : round(glob_rad, 2),   # best_ghi integrated
        "beam_rad_Whm2"  : round(beam_rad, 2),   # DNI*cos(zen) integrated
        "diff_rad_Whm2"  : round(diff_rad, 2),   # diffuse integrated
        "dw_solar_Whm2"  : round(dw_solar, 2),   # raw dw_solar (reference only)
    }

# ============================================================
# MAIN
# ============================================================

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    records = []
    total   = len(STATIONS) * len(YEARS) * len(TARGET_DOYS)
    count   = 0

    print(f"Processing {total} station-year-DOY combinations")
    print(f"SURFRAD dir : {SURFRAD_DIR}")
    print(f"Output dir  : {OUTPUT_DIR}")
    print()

    for code in STATIONS:
        for year in YEARS:
            yy = str(year)[2:]
            for doy in TARGET_DOYS:
                count += 1
                fname    = f"{code}{yy}{doy:03d}.dat"
                filepath = Path(SURFRAD_DIR) / code / str(year) / fname

                if not filepath.exists():
                    status = "NO FILE"
                else:
                    result = process_file(filepath, code, year, doy)
                    if result:
                        records.append(result)
                        status = f"OK  glob={result['glob_rad_Whm2']:.1f} beam={result['beam_rad_Whm2']:.1f} diff={result['diff_rad_Whm2']:.1f} Wh/m²"
                    else:
                        status = "SKIP (insufficient clear-sky data)"

                print(f"[{count:4d}/{total}] {code} {year} DOY{doy:03d} → {status}")

    if not records:
        print("\nERROR: No valid records found. Check SURFRAD_DIR path and file downloads.")
        return

    df = pd.DataFrame(records).sort_values(["station","doy","year"]).reset_index(drop=True)

    # --------------------------------------------------------
    # Output 1 — All valid daily records
    # --------------------------------------------------------
    out1 = Path(OUTPUT_DIR) / "surfrad_daily_clearsky.csv"
    df.to_csv(out1, index=False)
    print(f"\nSaved: {out1}  ({len(df)} records)")

    # --------------------------------------------------------
    # Output 2 — Multi-year means per station x DOY
    # This is what you compare against r.sun pixel values
    # --------------------------------------------------------
    agg = df.groupby(
        ["station","station_name","state","lat","lon","elev_m","doy"]
    ).agg(
        n_years            = ("year",           "count"),
        glob_rad_mean      = ("glob_rad_Whm2",  "mean"),
        glob_rad_std       = ("glob_rad_Whm2",  "std"),
        glob_rad_min       = ("glob_rad_Whm2",  "min"),
        glob_rad_max       = ("glob_rad_Whm2",  "max"),
        beam_rad_mean      = ("beam_rad_Whm2",  "mean"),
        beam_rad_std       = ("beam_rad_Whm2",  "std"),
        diff_rad_mean      = ("diff_rad_Whm2",  "mean"),
        diff_rad_std       = ("diff_rad_Whm2",  "std"),
        clearsky_frac_mean = ("clearsky_frac",  "mean"),
        n_clearsky_mean    = ("n_clearsky_min", "mean"),
    ).reset_index()

    # Round numeric columns
    num_cols = [c for c in agg.columns if agg[c].dtype == float]
    agg[num_cols] = agg[num_cols].round(2)

    out2 = Path(OUTPUT_DIR) / "surfrad_multiyear_means.csv"
    agg.to_csv(out2, index=False)
    print(f"Saved: {out2}  ({len(agg)} station-DOY rows)")

    # --------------------------------------------------------
    # Output 3 — Data availability matrix
    # --------------------------------------------------------
    avail = df.pivot_table(
        index="doy",
        columns="station",
        values="glob_rad_Whm2",
        aggfunc="count"
    ).fillna(0).astype(int)

    out3 = Path(OUTPUT_DIR) / "surfrad_data_availability.csv"
    avail.to_csv(out3)
    print(f"Saved: {out3}")

    # --------------------------------------------------------
    # Console summary
    # --------------------------------------------------------
    print(f"""
=== COMPLETE ===
Valid records    : {len(df)}
Stations         : {df['station'].nunique()} / {len(STATIONS)}
Years covered    : {df['year'].min()}–{df['year'].max()}
DOYs covered     : {sorted(df['doy'].unique())}

Per-station record counts:
{df.groupby('station')['year'].count().to_string()}

Output files ready for R:
  surfrad_daily_clearsky.csv    — full record, all years
  surfrad_multiyear_means.csv   — DOY means per station (use for r.sun comparison)
  surfrad_data_availability.csv — coverage matrix
""")


if __name__ == "__main__":
    main()

Processing 840 station-year-DOY combinations
SURFRAD dir : C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\surfrad_data
Output dir  : C:\Users\mgvhy\OneDrive - University of Missouri\scientific_data\analysis\surfrad_processed_0.35

[   1/840] bon 2015 DOY015 → OK  glob=2853.3 beam=2220.8 diff=632.5 Wh/m²
[   2/840] bon 2015 DOY045 → OK  glob=1938.3 beam=1489.6 diff=448.6 Wh/m²
[   3/840] bon 2015 DOY074 → OK  glob=4034.3 beam=3405.1 diff=629.2 Wh/m²
[   4/840] bon 2015 DOY105 → OK  glob=5710.0 beam=5016.7 diff=693.3 Wh/m²
[   5/840] bon 2015 DOY135 → OK  glob=3353.0 beam=2441.5 diff=911.5 Wh/m²
[   6/840] bon 2015 DOY166 → OK  glob=762.4 beam=536.7 diff=225.7 Wh/m²
[   7/840] bon 2015 DOY196 → OK  glob=1517.0 beam=1058.5 diff=458.4 Wh/m²
[   8/840] bon 2015 DOY227 → OK  glob=6005.0 beam=4762.0 diff=1243.0 Wh/m²
[   9/840] bon 2015 DOY258 → OK  glob=6042.3 beam=5334.1 diff=708.2 Wh/m²
[  10/840] bon 2015 DOY288 → OK  glob=1571.8 beam=1186.9 diff=384.9 Wh/m²
[  